# AnemiaFusionNet — Phase 5: Training Strategy

**Project:** AI-Powered Multimodal Anemia Detection System  
**Phase:** 5 — Full Pipeline Training  
**Builds on:** Phase 2 (preprocessing) · Phase 3 (feature extractors) · Phase 4 (transformer fusion)

This notebook implements a **three-stage progressive training strategy** for the complete multimodal anemia-detection pipeline:

| Stage | Frozen | Trainable | LR |
|-------|--------|-----------|-----|
| 1 — Fusion warm-up | All extractors | Transformer + Classifier | 1e-4 |
| 2 — Partial fine-tune | Early EfficientNet layers | Last 2 blocks · Clinical · Geo · Transformer | 5e-5 |
| 3 — End-to-end | Nothing | Everything (+ gradient clipping) | 1e-5 |

**Checkpoints required from previous phases:**
```
models/image_feature_extractor.pth
models/clinical_feature_extractor.pth
models/geo_feature_extractor.pth
models/multimodal_transformer.pth
```

---

### Contents
1. Imports & Setup  
2. Configuration  
3. Load Phase 2 Data  
4. Dataset & DataLoaders  
5. Load Pretrained Models  
6. Build Full Model  
7. Training Strategy (3 Stages)  
8. Loss & Class Imbalance  
9. Training Loop  
10. Evaluation  
11. Visualisations  
12. Final Verification  


---
## Section 1 — Imports & Setup

In [1]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test_true, y_test_prob)
print(f"AUC-ROC = {auc:.4f}")

Error: No connection selected.

In [ ]:
# ── Auto-install tqdm if not available ────────────────────────────────────────
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_ensure("tqdm")

# ── Standard library ──────────────────────────────────────────────────────────
import os
import copy
import time
import random
import warnings
warnings.filterwarnings("ignore")

# ── Numerical / data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")          # non-interactive backend — safe on all machines
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── Torchvision ───────────────────────────────────────────────────────────────
import torchvision.models as tv_models
from torchvision import transforms

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
    matthews_corrcoef,
)

# ── Progress bars ─────────────────────────────────────────────────────────────
from tqdm.auto import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
from pathlib import Path

print(f"PyTorch   : {torch.__version__}")
print(f"Device    : {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("All imports OK ✓")


PyTorch   : 2.12.1+cpu
Device    : cpu
All imports OK ✓


In [ ]:
# ── Project root (same auto-detection as Phases 2-4) ─────────────────────────
_cwd = Path.cwd()
if   (_cwd / "dataset").exists():           PROJECT_ROOT = _cwd
elif (_cwd.parent / "dataset").exists():    PROJECT_ROOT = _cwd.parent
else:                                        PROJECT_ROOT = _cwd   # graceful fallback

DATASET_DIR   = PROJECT_ROOT / "dataset"
PROCESSED_DIR = DATASET_DIR  / "processed"
IMAGE_DIR     = DATASET_DIR  / "images"
MODELS_DIR    = PROJECT_ROOT / "models"
OUTPUTS_DIR   = PROJECT_ROOT / "outputs"
REPORTS_DIR   = PROJECT_ROOT / "reports"

for _d in [MODELS_DIR, OUTPUTS_DIR, REPORTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")


Project root : C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet
Device       : cpu


---
## Section 2 — Configuration

Single source of truth for all hyper-parameters. Downstream cells read only from `CONFIG`.

In [ ]:
CONFIG = {
    # ── Data ──────────────────────────────────────────────────────────────────
    "batch_size"           : 16,
    "seed"                 : 42,
    "train_ratio"          : 0.70,
    "val_ratio"            : 0.15,
    # test_ratio = 1 - train - val = 0.15

    # ── Training stages ───────────────────────────────────────────────────────
    "stage1_epochs"        : 10,
    "stage2_epochs"        : 10,
    "stage3_epochs"        : 10,

    # ── Learning rates ────────────────────────────────────────────────────────
    "lr_stage1"            : 1e-4,
    "lr_stage2"            : 5e-5,
    "lr_stage3"            : 1e-5,

    # ── Regularisation ────────────────────────────────────────────────────────
    "weight_decay"         : 1e-5,
    "grad_clip"            : 1.0,        # max norm for Stage 3 clipping

    # ── Early stopping ────────────────────────────────────────────────────────
    "early_stop_patience"  : 5,
    "monitor_metric"       : "f1",       # metric watched by early stopping

    # ── Architecture (used when creating fresh demo checkpoints only) ─────────
    "image_dim"            : 1280,       # EfficientNet-B0 output
    "clinical_dim"         : 64,         # ClinicalFeatureExtractor output
    "geo_dim"              : 64,         # GeoRiskFeatureExtractor output
    "latent_dim"           : 128,
    "num_heads"            : 4,
    "num_layers"           : 2,
    "ffn_dim"              : 256,
    "dropout"              : 0.1,
}

# Reproducibility
SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k:<24} : {v}")


Configuration:
  batch_size               : 16
  seed                     : 42
  train_ratio              : 0.7
  val_ratio                : 0.15
  stage1_epochs            : 10
  stage2_epochs            : 10
  stage3_epochs            : 10
  lr_stage1                : 0.0001
  lr_stage2                : 5e-05
  lr_stage3                : 1e-05
  weight_decay             : 1e-05
  grad_clip                : 1.0
  early_stop_patience      : 5
  monitor_metric           : f1
  image_dim                : 1280
  clinical_dim             : 64
  geo_dim                  : 64
  latent_dim               : 128
  num_heads                : 4
  num_layers               : 2
  ffn_dim                  : 256
  dropout                  : 0.1


---
## Section 3 — Load Phase 2 Data

Load `clinical_processed.csv` and `geo_processed.csv` generated by Phase 2.

**Hard requirement (spec):** after dropping non-feature columns, `clinical_input_dim` must equal 4.  
If the dataset has a different number of features a `RuntimeError` is raised immediately with a
clear explanation, rather than letting a shape mismatch surface later as a cryptic matrix-multiply
error.


In [ ]:
# ── 3.1  Load CSVs ────────────────────────────────────────────────────────────
CLINICAL_CSV = PROCESSED_DIR / "clinical_processed.csv"
GEO_CSV      = PROCESSED_DIR / "geo_processed.csv"

REQUIRED_CLINICAL_DIM = 4   # Phase 3 was trained with 4 clinical features

# ── Helper: remove unwanted columns consistently ──────────────────────────────
def _clean_df(df: pd.DataFrame, extra_drop=()) -> pd.DataFrame:
    """Drop Unnamed:*, Patient_ID, Note, and any caller-specified extras."""
    drop = [c for c in df.columns
            if c.startswith("Unnamed:") or c in ("Patient_ID", "Note") or c in extra_drop]
    return df.drop(columns=drop)

# ── Clinical ──────────────────────────────────────────────────────────────────
if CLINICAL_CSV.exists():
    clinical_df_raw = pd.read_csv(CLINICAL_CSV)
    clinical_df_raw = _clean_df(clinical_df_raw)
    print(f"Clinical CSV  : {CLINICAL_CSV}  {clinical_df_raw.shape}")
    HAS_CLINICAL = True
else:
    clinical_df_raw = None
    HAS_CLINICAL = False
    print("WARNING: clinical_processed.csv not found — synthetic data will be used.")

# ── Geo ───────────────────────────────────────────────────────────────────────
if GEO_CSV.exists():
    geo_df_raw = pd.read_csv(GEO_CSV)
    geo_df_raw = _clean_df(geo_df_raw)
    print(f"Geo CSV       : {GEO_CSV}  {geo_df_raw.shape}")
    HAS_GEO = True
else:
    geo_df_raw = None
    HAS_GEO = False
    print("WARNING: geo_processed.csv not found — synthetic geo risk will be used.")


Clinical CSV  : C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\dataset\processed\clinical_processed.csv  (95, 5)
Geo CSV       : C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\dataset\processed\geo_processed.csv  (95, 4)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 3.2 Identify label column and feature columns
# ──────────────────────────────────────────────────────────────────────────────

LABEL_COL = "Label"

if HAS_CLINICAL:

    # ----------------------------------------------------------
    # If Label does not exist, create it from Hgb
    # ----------------------------------------------------------
    if LABEL_COL not in clinical_df_raw.columns:

        print("Label column not found.")
        print("Creating Label from Hgb values...")

        if "Hgb" not in clinical_df_raw.columns:
            raise RuntimeError(
                "Cannot create Label because Hgb column is missing."
            )

        # IMPORTANT:
        # Use ORIGINAL Hgb values BEFORE scaling if available.
        # If your Hgb is already standardized this is only a temporary fix.
        clinical_df_raw[LABEL_COL] = (
            clinical_df_raw["Hgb"] < 12
        ).astype(int)

    # ----------------------------------------------------------
    # Required feature columns
    # ----------------------------------------------------------
    feature_cols = [
        "Hgb",
        "Age",
        "Gender_F",
        "Gender_M"
    ]

    # Verify all required columns exist
    missing = [c for c in feature_cols if c not in clinical_df_raw.columns]

    if len(missing) > 0:
        raise RuntimeError(
            f"Missing required clinical features: {missing}"
        )

    clinical_features_df = clinical_df_raw[feature_cols]
    labels_series = clinical_df_raw[LABEL_COL]

    clinical_input_dim = clinical_features_df.shape[1]

    print(f"\nClinical feature columns ({clinical_input_dim})")
    print(feature_cols)

    if clinical_input_dim != REQUIRED_CLINICAL_DIM:
        raise RuntimeError(
            f"Expected {REQUIRED_CLINICAL_DIM} features "
            f"but found {clinical_input_dim}."
        )

    labels = labels_series.astype(int).values

    print("\nClinical input dimension:", clinical_input_dim)
    print("Label distribution:")
    print(dict(zip(*np.unique(labels, return_counts=True))))

else:

    raise RuntimeError(
        "Clinical dataset not found."
    )


Clinical feature columns (4)
['Hgb', 'Age', 'Gender_F', 'Gender_M']

Clinical input dimension: 4
Label distribution:
{np.int64(0): np.int64(39), np.int64(1): np.int64(56)}


In [ ]:
# ── 3.3  Geo risk column ──────────────────────────────────────────────────────
if HAS_GEO and "State_Risk" in geo_df_raw.columns:
    if len(geo_df_raw) == len(clinical_features_df):
        state_risk = geo_df_raw["State_Risk"].values.astype(np.float32)
    else:
        state_risk = np.full(len(clinical_features_df),
                             float(geo_df_raw["State_Risk"].iloc[0]),
                             dtype=np.float32)
else:
    state_risk = np.full(len(clinical_features_df), 0.534, dtype=np.float32)

print(f"State_Risk range : [{state_risk.min():.3f}, {state_risk.max():.3f}]")
print(f"Total samples    : {len(clinical_features_df)}")


State_Risk range : [0.534, 0.534]
Total samples    : 95


---
## Section 4 — Dataset & DataLoaders

Unified `AnemiaDataset` returning `(image, clinical, geo, label)`. Split is 70 / 15 / 15, stratified.

In [ ]:
# ── 4.1  Image transforms ─────────────────────────────────────────────────────
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])


In [ ]:
# ── 4.2  Dataset class ────────────────────────────────────────────────────────
class AnemiaDataset(Dataset):
    """Returns (image_tensor, clinical_tensor, geo_tensor, label) per sample.

    Images are loaded from IMAGE_DIR/{anemic,non_anemic}/ if present;
    otherwise a deterministic synthetic tensor is used so training proceeds
    without raw image files.
    """

    def __init__(self,
                 clinical_arr : np.ndarray,
                 geo_arr      : np.ndarray,
                 label_arr    : np.ndarray,
                 image_dir    : Path,
                 transform    = None):
        self.clinical   = clinical_arr.astype(np.float32)
        self.geo        = geo_arr.astype(np.float32).reshape(-1, 1)
        self.labels     = label_arr.astype(np.float32)
        self.image_dir  = image_dir
        self.transform  = transform or eval_transform

        # Index real images by class folder
        self.class_images = {0: [], 1: []}
        if image_dir.exists():
            for cls, folder in [(0, "non_anemic"), (1, "anemic")]:
                p = image_dir / folder
                if p.exists():
                    self.class_images[cls] = sorted(p.glob("*.*"))
        self.has_images = any(len(v) > 0 for v in self.class_images.values())

    def __len__(self):
        return len(self.labels)

    def _get_image(self, label: int, idx: int) -> torch.Tensor:
        imgs = self.class_images.get(int(label), [])
        if self.has_images and imgs:
            from PIL import Image as PILImage
            img = PILImage.open(imgs[idx % len(imgs)]).convert("RGB")
            return self.transform(img)
        # Deterministic synthetic fallback
        g = torch.Generator().manual_seed(idx)
        return torch.randn(3, 224, 224, generator=g)

    def __getitem__(self, idx):
        label = self.labels[idx]
        return (
            self._get_image(int(label), idx),
            torch.from_numpy(self.clinical[idx]),
            torch.from_numpy(self.geo[idx]),
            torch.tensor(label, dtype=torch.float32),
        )


In [ ]:
# ── 4.3  Train / val / test split ─────────────────────────────────────────────
_clin_arr = clinical_features_df.values.astype(np.float32)
_geo_arr  = state_risk
_lab_arr  = labels

# 70 / 15 / 15  stratified
_idx = np.arange(len(_lab_arr))
train_idx, temp_idx = train_test_split(_idx, test_size=0.30, random_state=SEED, stratify=_lab_arr)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.50, random_state=SEED, stratify=_lab_arr[temp_idx])

print(f"Split — Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

train_dataset = AnemiaDataset(_clin_arr[train_idx], _geo_arr[train_idx], _lab_arr[train_idx], IMAGE_DIR, train_transform)
val_dataset   = AnemiaDataset(_clin_arr[val_idx],   _geo_arr[val_idx],   _lab_arr[val_idx],   IMAGE_DIR, eval_transform)
test_dataset  = AnemiaDataset(_clin_arr[test_idx],  _geo_arr[test_idx],  _lab_arr[test_idx],  IMAGE_DIR, eval_transform)

print(f"Images found on disk : {train_dataset.has_images}")


Split — Train: 66  Val: 14  Test: 15
Images found on disk : False


In [ ]:
# ── 4.4  DataLoaders  (WeightedRandomSampler for training) ───────────────────
_train_labels = _lab_arr[train_idx]
_counts       = np.bincount(_train_labels.astype(int))
_weights      = (1.0 / np.maximum(_counts, 1))[_train_labels.astype(int)]
_sampler      = WeightedRandomSampler(weights=_weights, num_samples=len(_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], sampler=_sampler,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}  Test batches: {len(test_loader)}")


Train batches: 5  Val batches: 1  Test batches: 1


In [ ]:
images, clinical, geo, labels = next(iter(train_loader))

print(torch.isnan(images).any())
print(torch.isnan(clinical).any())
print(torch.isnan(geo).any())

tensor(False)
tensor(False)
tensor(False)


---
## Section 5 — Load Pretrained Models

All four checkpoint files are loaded using **their own saved metadata** — no architecture
dimensions are hard-coded here. The loader validates:

- Required top-level keys exist (`state_dict`, `metadata`/`config`)
- All architectural keys needed for reconstruction are present in metadata
- No NaN values in any loaded weight tensor


In [ ]:
# ── 5.1  Architecture class definitions (must match Phases 3 & 4 exactly) ─────

# ─── Image Feature Extractor ──────────────────────────────────────────────────
class ImageFeatureExtractor(nn.Module):
    """EfficientNet-B0 (or ResNet-50) backbone with classifier head removed."""

    SUPPORTED = {
        "EfficientNet-B0": {
            "builder"     : lambda w: tv_models.efficientnet_b0(weights=w),
            "weights_cls" : tv_models.EfficientNet_B0_Weights.DEFAULT,
            "feature_dim" : 1280,
        },
        "ResNet-50": {
            "builder"     : lambda w: tv_models.resnet50(weights=w),
            "weights_cls" : tv_models.ResNet50_Weights.DEFAULT,
            "feature_dim" : 2048,
        },
    }

    def __init__(self, backbone_name: str = "EfficientNet-B0", pretrained: bool = True):
        super().__init__()
        spec    = self.SUPPORTED[backbone_name]
        weights = spec["weights_cls"] if pretrained else None
        base    = spec["builder"](weights)

        if backbone_name == "EfficientNet-B0":
            for p in base.parameters(): p.requires_grad = False
            for block in list(base.features.children())[-2:]:
                for p in block.parameters(): p.requires_grad = True
            self.backbone = base.features
            self.pool     = base.avgpool
        else:  # ResNet-50
            for p in base.parameters(): p.requires_grad = False
            for layer in [base.layer3, base.layer4]:
                for p in layer.parameters(): p.requires_grad = True
            self.backbone = nn.Sequential(*list(base.children())[:-1])
            self.pool     = nn.Identity()

        self.backbone_name = backbone_name
        self.feature_dim   = spec["feature_dim"]

    def forward(self, x):
        return self.pool(self.backbone(x)).flatten(1)


# ─── Clinical Feature Extractor ───────────────────────────────────────────────
class ClinicalFeatureExtractor(nn.Module):
    """MLP: input_dim → 256 → 128 → 64."""

    def __init__(self, input_dim: int, dropout: float = 0.3):
        super().__init__()
        self.input_dim  = input_dim
        self.output_dim = 64
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, 64),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x): return self.network(x)


# ─── Geo Risk Feature Extractor ───────────────────────────────────────────────
class GeoRiskFeatureExtractor(nn.Module):
    """GeoNet: 1 → 32 → 64."""

    def __init__(self):
        super().__init__()
        self.input_dim  = 1
        self.output_dim = 64
        self.network = nn.Sequential(
            nn.Linear(1, 32),  nn.ReLU(inplace=True),
            nn.Linear(32, 64), nn.ReLU(inplace=True),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x): return self.network(x)


# ─── Multimodal Transformer ───────────────────────────────────────────────────
class MultiModalTransformer(nn.Module):
    """
    image(image_dim) + clinical(clinical_dim) + geo(geo_dim)
    → projection(latent_dim each) → stack(B,3,latent_dim)
    → TransformerEncoder → flatten → classifier → logit
    """

    def __init__(self, image_dim=1280, clinical_dim=64, geo_dim=64,
                 latent_dim=128, num_heads=4, num_layers=2, ffn_dim=256, dropout=0.1):
        super().__init__()
        self.image_proj    = nn.Linear(image_dim,    latent_dim)
        self.clinical_proj = nn.Linear(clinical_dim, latent_dim)
        self.geo_proj      = nn.Linear(geo_dim,      latent_dim)

        enc = nn.TransformerEncoderLayer(
            d_model=latent_dim, nhead=num_heads, dim_feedforward=ffn_dim,
            dropout=dropout, batch_first=True, activation="relu",
        )
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)

        fused = latent_dim * 3
        self.classifier = nn.Sequential(
            nn.Linear(fused, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, 64),    nn.ReLU(inplace=True),
            nn.Linear(64, 1),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, img_f, clin_f, geo_f):
        tokens = torch.stack(
            [self.image_proj(img_f), self.clinical_proj(clin_f), self.geo_proj(geo_f)], dim=1
        )
        return self.classifier(self.transformer(tokens).flatten(1))


print("Architecture classes defined: ImageFeatureExtractor, ClinicalFeatureExtractor, "
      "GeoRiskFeatureExtractor, MultiModalTransformer")


Architecture classes defined: ImageFeatureExtractor, ClinicalFeatureExtractor, GeoRiskFeatureExtractor, MultiModalTransformer


In [ ]:
# ── 5.2  Generic NaN validator ────────────────────────────────────────────────
def _check_no_nans(model: nn.Module, name: str) -> None:
    """Raise RuntimeError if any weight tensor in `model` contains NaN values."""
    bad = [(n, p) for n, p in model.named_parameters() if torch.isnan(p).any()]
    if bad:
        details = "\n".join(f"  {n}: {torch.isnan(p).sum().item()} NaN(s)" for n, p in bad)
        raise RuntimeError(
            f"NaN weights detected in {name} after loading checkpoint:\n{details}\n"
            f"The checkpoint may be corrupt. Re-run Phase 3 to regenerate it."
        )
    print(f"  NaN check passed ✓  ({name})")


# ── 5.3  Checkpoint loader utilities ─────────────────────────────────────────
def _load_raw(path: Path, label: str) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"[{label}] Checkpoint not found: {path}")
    try:
        return torch.load(path, map_location="cpu")
    except Exception as e:
        raise RuntimeError(f"[{label}] Failed to deserialise {path}: {e}") from e


def _meta(ckpt: dict, label: str) -> dict:
    """Return the architecture dict, accepting both 'metadata' and 'config' keys."""
    for key in ("metadata", "config"):
        if key in ckpt:
            return ckpt[key]
    raise RuntimeError(
        f"[{label}] Checkpoint is missing both 'metadata' and 'config' keys.\n"
        f"  Found keys: {list(ckpt.keys())}"
    )


def _load_state(model, ckpt, label):

    state_dict = ckpt["state_dict"].copy()

    # ----------------------------------------------------------
    # Backward compatibility for older Phase 4 checkpoints
    # ----------------------------------------------------------
    rename = {
        "image_projection": "image_proj",
        "clinical_projection": "clinical_proj",
        "geo_projection": "geo_proj",
    }

    new_state = {}

    for k, v in state_dict.items():

        new_key = k

        for old, new in rename.items():
            if k.startswith(old):
                new_key = k.replace(old, new, 1)

        new_state[new_key] = v

    missing, unexpected = model.load_state_dict(
        new_state,
        strict=False
    )

    if unexpected:
        print("Unexpected keys:", unexpected)

    if missing:
        print("Missing keys:", missing)

    print(f"✓ {label} weights loaded successfully.")


In [ ]:
# ── 5.4  Checkpoint paths ─────────────────────────────────────────────────────
IMAGE_CKPT_PATH    = MODELS_DIR / "image_feature_extractor.pth"
CLINICAL_CKPT_PATH = MODELS_DIR / "clinical_feature_extractor.pth"
GEO_CKPT_PATH      = MODELS_DIR / "geo_feature_extractor.pth"
TRANSFORMER_PATH   = MODELS_DIR / "multimodal_transformer.pth"

for _p in [IMAGE_CKPT_PATH, CLINICAL_CKPT_PATH, GEO_CKPT_PATH, TRANSFORMER_PATH]:
    print(f"  {'FOUND  ' if _p.exists() else 'MISSING'} {_p}")


  FOUND   C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\models\image_feature_extractor.pth
  FOUND   C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\models\clinical_feature_extractor.pth
  FOUND   C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\models\geo_feature_extractor.pth
  FOUND   C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\models\multimodal_transformer.pth


In [ ]:
# =============================================================================
# 5.5 Verify Required Checkpoints
# =============================================================================

IMAGE_CKPT_PATH = MODELS_DIR / "image_feature_extractor.pth"
CLINICAL_CKPT_PATH = MODELS_DIR / "clinical_feature_extractor.pth"
GEO_CKPT_PATH = MODELS_DIR / "geo_feature_extractor.pth"
TRANSFORMER_CKPT_PATH = MODELS_DIR / "multimodal_transformer.pth"

assert IMAGE_CKPT_PATH.exists(), f"Missing: {IMAGE_CKPT_PATH}"
assert CLINICAL_CKPT_PATH.exists(), f"Missing: {CLINICAL_CKPT_PATH}"
assert GEO_CKPT_PATH.exists(), f"Missing: {GEO_CKPT_PATH}"
assert TRANSFORMER_CKPT_PATH.exists(), f"Missing: {TRANSFORMER_CKPT_PATH}"

print("✅ All required checkpoints found.")

✅ All required checkpoints found.


In [ ]:
# ── 5.6  Load image extractor ─────────────────────────────────────────────────
_ckpt_img  = _load_raw(IMAGE_CKPT_PATH, "Image")
_meta_img  = _meta(_ckpt_img, "Image")

if "backbone" not in _meta_img or "feature_dim" not in _meta_img:
    raise RuntimeError(f"Image checkpoint metadata missing 'backbone' or 'feature_dim'. Found: {_meta_img}")

image_extractor = ImageFeatureExtractor(backbone_name=_meta_img["backbone"], pretrained=False).to(DEVICE)
_load_state(image_extractor, _ckpt_img, "Image")
_check_no_nans(image_extractor, "ImageFeatureExtractor")
print(f"[Image] OK — backbone={_meta_img['backbone']}, feature_dim={_meta_img['feature_dim']}")

# ── 5.7  Load clinical extractor ──────────────────────────────────────────────
_ckpt_clin  = _load_raw(CLINICAL_CKPT_PATH, "Clinical")
_meta_clin  = _meta(_ckpt_clin, "Clinical")

if "input_dim" not in _meta_clin:
    raise RuntimeError(f"Clinical checkpoint metadata missing 'input_dim'. Found: {_meta_clin}")

_ckpt_dim = int(_meta_clin["input_dim"])
if _ckpt_dim != clinical_input_dim:
    raise RuntimeError(
        f"Clinical checkpoint expects {_ckpt_dim} features but dataset has {clinical_input_dim}.\n"
        f"Regenerate Phase 3 with the current dataset or adjust Phase 2 preprocessing."
    )

clinical_extractor = ClinicalFeatureExtractor(input_dim=_ckpt_dim).to(DEVICE)
_load_state(clinical_extractor, _ckpt_clin, "Clinical")
_check_no_nans(clinical_extractor, "ClinicalFeatureExtractor")
print(f"[Clinical] OK — input_dim={_ckpt_dim}, output_dim={_meta_clin.get('output_dim', 64)}")

# ── 5.8  Load geo extractor ───────────────────────────────────────────────────
_ckpt_geo  = _load_raw(GEO_CKPT_PATH, "Geo")
# GeoRiskFeatureExtractor is always 1-in / 64-out; metadata is informational only
geo_extractor = GeoRiskFeatureExtractor().to(DEVICE)
_load_state(geo_extractor, _ckpt_geo, "Geo")
_check_no_nans(geo_extractor, "GeoRiskFeatureExtractor")
print(f"[Geo] OK — input_dim=1, output_dim=64")

# ── 5.9  Load transformer ─────────────────────────────────────────────────────
_ckpt_tfm  = _load_raw(TRANSFORMER_PATH, "Transformer")
_meta_tfm  = _meta(_ckpt_tfm, "Transformer")

_tfm_keys = ("image_dim","clinical_dim","geo_dim","latent_dim","num_heads","num_layers","ffn_dim","dropout")
_missing  = [k for k in _tfm_keys if k not in _meta_tfm]
if _missing:
    raise RuntimeError(f"Transformer checkpoint metadata missing keys: {_missing}.  Found: {list(_meta_tfm.keys())}")

transformer = MultiModalTransformer(**{k: _meta_tfm[k] for k in _tfm_keys}).to(DEVICE)
_load_state(transformer, _ckpt_tfm, "Transformer")
_check_no_nans(transformer, "MultiModalTransformer")
print(f"[Transformer] OK — config={dict((k,_meta_tfm[k]) for k in _tfm_keys)}")

print("\n✓ All four checkpoints loaded and validated.")


✓ Image weights loaded successfully.
  NaN check passed ✓  (ImageFeatureExtractor)
[Image] OK — backbone=EfficientNet-B0, feature_dim=1280
✓ Clinical weights loaded successfully.
  NaN check passed ✓  (ClinicalFeatureExtractor)
[Clinical] OK — input_dim=4, output_dim=64
✓ Geo weights loaded successfully.
  NaN check passed ✓  (GeoRiskFeatureExtractor)
[Geo] OK — input_dim=1, output_dim=64
✓ Transformer weights loaded successfully.
  NaN check passed ✓  (MultiModalTransformer)
[Transformer] OK — config={'image_dim': 1280, 'clinical_dim': 64, 'geo_dim': 64, 'latent_dim': 128, 'num_heads': 4, 'num_layers': 2, 'ffn_dim': 256, 'dropout': 0.1}

✓ All four checkpoints loaded and validated.


---
## Section 6 — Build Full Model

`MultimodalAnemiaModel` wires the four pretrained components into a single `nn.Module`.

In [ ]:
class MultimodalAnemiaModel(nn.Module):
    """
    Wraps all four Phase-3/4 components into one module.

    forward() passes each modality through its extractor, then feeds
    the resulting feature vectors to the transformer for fusion.

    freeze_backbone() / unfreeze_backbone() control which parameters
    require gradients across the three training stages.
    """

    def __init__(self,
                 image_ext    : nn.Module,
                 clinical_ext : nn.Module,
                 geo_ext      : nn.Module,
                 transformer  : nn.Module):
        super().__init__()
        self.image_ext    = image_ext
        self.clinical_ext = clinical_ext
        self.geo_ext      = geo_ext
        self.transformer  = transformer

    # ── Freeze / unfreeze helpers ─────────────────────────────────────────────
    def freeze_backbone(self) -> None:
        """Stage 1: freeze all three feature extractors completely."""
        for ext in [self.image_ext, self.clinical_ext, self.geo_ext]:
            for p in ext.parameters():
                p.requires_grad = False

    def unfreeze_backbone(self, mode: str = "partial") -> None:
        """
        Stage 2 (mode='partial'): unfreeze last 2 EfficientNet blocks,
                                   clinical extractor, geo extractor.
        Stage 3 (mode='full')   : unfreeze everything.
        """
        if mode == "full":
            for ext in [self.image_ext, self.clinical_ext, self.geo_ext]:
                for p in ext.parameters():
                    p.requires_grad = True
        elif mode == "partial":
            # Clinical and geo — fully unfreeze
            for ext in [self.clinical_ext, self.geo_ext]:
                for p in ext.parameters():
                    p.requires_grad = True
            # Image — only last 2 EfficientNet feature blocks
            if hasattr(self.image_ext, "backbone"):
                children = list(self.image_ext.backbone.children())
                for block in children[-2:]:
                    for p in block.parameters():
                        p.requires_grad = True
        else:
            raise ValueError(f"Unknown unfreeze mode '{mode}'. Use 'partial' or 'full'.")

    def trainable_params(self):
        return [p for p in self.parameters() if p.requires_grad]

    # ── Forward pass ─────────────────────────────────────────────────────────
    def forward(self, images, clinical, geo):
        img_f  = self.image_ext(images)
        clin_f = self.clinical_ext(clinical)
        geo_f  = self.geo_ext(geo)
        return self.transformer(img_f, clin_f, geo_f)   # raw logit (B,1)


# ── Instantiate ───────────────────────────────────────────────────────────────
model = MultimodalAnemiaModel(image_extractor, clinical_extractor, geo_extractor, transformer).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("=" * 55)
print("  MULTIMODAL ANEMIA MODEL")
print("=" * 55)
print(f"  Total params     : {total:,}")
print(f"  Trainable params : {trainable:,}  ({100*trainable/total:.1f}%)")
print("=" * 55)


  MULTIMODAL ANEMIA MODEL
  Total params     : 4,612,797
  Trainable params : 1,734,641  (37.6%)


---
## Section 7 — Training Strategy

Three progressive stages. Each stage builds an optimizer over the currently-trainable
parameters and a `CosineAnnealingLR` scheduler.

| Stage | Frozen | Trainable | LR |
|-------|--------|-----------|-----|
| 1 | All extractors | Transformer + classifier | 1e-4 |
| 2 | Early EfficientNet layers | Last 2 blocks · Clinical · Geo · Transformer | 5e-5 |
| 3 | Nothing | Everything + gradient clipping | 1e-5 |


In [ ]:
def build_stage_optimizer(model: MultimodalAnemiaModel, lr: float):
    """Build an Adam optimizer over the model's currently-trainable parameters."""
    return optim.Adam(model.trainable_params(), lr=lr, weight_decay=CONFIG["weight_decay"])

def build_scheduler(optimizer, epochs: int):
    return CosineAnnealingLR(optimizer, T_max=epochs)


print("Stage setup functions ready.")
print("Stages will be initialised at the start of each training section.")


Stage setup functions ready.
Stages will be initialised at the start of each training section.


---
## Section 8 — Loss & Class Imbalance

`BCEWithLogitsLoss` with `pos_weight` derived from the training label distribution.

In [ ]:
# ── 8.1  Compute pos_weight from training labels ──────────────────────────────
_train_y = _lab_arr[train_idx]
_pos     = int(_train_y.sum())
_neg     = int(len(_train_y) - _pos)

if _pos > 0 and _neg > 0:
    pos_weight = torch.tensor([_neg / _pos], dtype=torch.float32).to(DEVICE)
    print(f"pos_weight = {pos_weight.item():.4f}  (neg={_neg} / pos={_pos})")
else:
    pos_weight = torch.ones(1, device=DEVICE)
    print("WARNING: only one class in training labels — pos_weight set to 1.0; "
          "using WeightedRandomSampler as fallback (already applied in DataLoader).")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Loss : BCEWithLogitsLoss  pos_weight={pos_weight.item():.4f}")


pos_weight = 0.6923  (neg=27 / pos=39)
Loss : BCEWithLogitsLoss  pos_weight=0.6923


---
## Section 9 — Training Loop

Reusable `train_one_epoch()`, `validate()`, and `test()` functions with NaN guards, tqdm progress bars, early stopping on validation F1, and automatic best-model checkpointing.

In [ ]:
# ── 9.1  Metric helper ────────────────────────────────────────────────────────
def _metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold=0.5) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {
        "accuracy"   : accuracy_score(y_true, y_pred),
        "precision"  : precision_score(y_true, y_pred, zero_division=0),
        "recall"     : recall_score(y_true, y_pred, zero_division=0),
        "f1"         : f1_score(y_true, y_pred, zero_division=0),
        "roc_auc"    : auc,
        "specificity": tn/(tn+fp) if (tn+fp)>0 else 0.0,
        "sensitivity": tp/(tp+fn) if (tp+fn)>0 else 0.0,
    }


# ── 9.2  NaN guards ───────────────────────────────────────────────────────────
def _assert_no_nan_tensor(t: torch.Tensor, name: str) -> None:
    if torch.isnan(t).any():
        raise RuntimeError(f"NaN detected in {name}. Aborting.")

def _assert_no_nan_params(model: nn.Module, label: str) -> None:
    for n, p in model.named_parameters():
        if p.requires_grad and torch.isnan(p).any():
            raise RuntimeError(f"NaN in parameter '{n}' of {label} after optimizer step. Aborting.")


In [ ]:
# ── 9.3  train_one_epoch ─────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion,
                    stage: int, epoch: int, total_epochs: int) -> float:
    model.train()
    losses = []
    pbar = tqdm(loader, desc=f"Stage {stage}  Epoch {epoch}/{total_epochs}  [Train]", leave=False)

    for images, clinical, geo, labels in pbar:
        images   = images.to(DEVICE)
        clinical = clinical.to(DEVICE)
        geo      = geo.to(DEVICE)
        labels   = labels.to(DEVICE).unsqueeze(1)

        # ── NaN input check ──────────────────────────────────────────────────
        _assert_no_nan_tensor(images,   "images")
        _assert_no_nan_tensor(clinical, "clinical")
        _assert_no_nan_tensor(geo,      "geo")

        optimizer.zero_grad()
        logits = model(images, clinical, geo)
        _assert_no_nan_tensor(logits, "logits")

        loss = criterion(logits, labels.float())

        # ── NaN loss check ───────────────────────────────────────────────────
        if torch.isnan(loss):
            raise RuntimeError(
                f"NaN loss at Stage {stage}, Epoch {epoch}. "
                "Check inputs, labels, and pos_weight. Aborting."
            )

        loss.backward()

        # ── Stage 3: gradient clipping ───────────────────────────────────────
        if stage == 3:
            nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])

        optimizer.step()

        # ── Post-step NaN parameter check ────────────────────────────────────
        _assert_no_nan_params(model, "MultimodalAnemiaModel")

        losses.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return float(np.mean(losses))


In [ ]:
# ── 9.4  validate / test ──────────────────────────────────────────────────────
def _eval_loop(model, loader, criterion, split_name: str):
    model.eval()
    losses, y_true, y_prob = [], [], []

    with torch.no_grad():
        for images, clinical, geo, labels in tqdm(loader, desc=f"  [{split_name}]", leave=False):
            images   = images.to(DEVICE)
            clinical = clinical.to(DEVICE)
            geo      = geo.to(DEVICE)
            labels   = labels.to(DEVICE).unsqueeze(1)

            _assert_no_nan_tensor(images,   f"{split_name} images")
            _assert_no_nan_tensor(clinical, f"{split_name} clinical")
            _assert_no_nan_tensor(geo,      f"{split_name} geo")

            logits = model(images, clinical, geo)
            _assert_no_nan_tensor(logits, f"{split_name} logits")

            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            _assert_no_nan_tensor(torch.tensor(probs), f"{split_name} probabilities")

            losses.append(criterion(logits, labels.float()).item())
            y_prob.extend(probs)
            y_true.extend(labels.cpu().numpy().ravel())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    if len(y_true) == 0:
        raise RuntimeError(f"{split_name} loop produced zero samples — DataLoader is empty.")

    return float(np.mean(losses)), y_true, y_prob


def validate(model, loader, criterion):
    return _eval_loop(model, loader, criterion, "Val")

def test(model, loader, criterion):
    return _eval_loop(model, loader, criterion, "Test")


In [ ]:
# ── 9.5  EarlyStopping (monitors validation F1) ───────────────────────────────
class EarlyStopping:
    def __init__(self, patience: int, label: str = ""):
        self.patience   = patience
        self.label      = label
        self.best_val   = -1.0
        self.counter    = 0
        self.best_state = None
        self.stopped    = False

    def step(self, metric: float, model: nn.Module) -> bool:
        if metric > self.best_val:
            self.best_val   = metric
            self.counter    = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stopped = True
        return self.stopped

    def restore(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


In [ ]:
# ── 9.6  Checkpoint save helper ───────────────────────────────────────────────
def save_model_checkpoint(model, optimizer, scheduler, epoch, metrics, path, stage):
    torch.save({
        "state_dict"      : model.state_dict(),
        "metadata"        : {
            "stage"       : stage,
            "epoch"       : epoch,
            "clinical_input_dim": clinical_input_dim,
        },
        "optimizer_state" : optimizer.state_dict(),
        "scheduler_state" : scheduler.state_dict(),
        "epoch"           : epoch,
        "metrics"         : metrics,
    }, path)
    print(f"  ✓ Saved: {path.name}  ({path.stat().st_size/1024:,.1f} KB)")


In [ ]:
# ── 9.7  Epoch-level display ──────────────────────────────────────────────────
def _print_epoch(stage, epoch, total, train_loss, val_loss, metrics, lr):
    print(
        f"  Stage {stage}  Ep {epoch:>2}/{total}  "
        f"TrainLoss={train_loss:.4f}  ValLoss={val_loss:.4f}  "
        f"Acc={metrics['accuracy']:.3f}  P={metrics['precision']:.3f}  "
        f"R={metrics['recall']:.3f}  F1={metrics['f1']:.3f}  "
        f"AUC={metrics['roc_auc']:.3f}  LR={lr:.1e}"
    )


# ── Storage for training history ──────────────────────────────────────────────
history = []          # list of per-epoch dicts

print("Training utilities ready.")


Training utilities ready.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 9.8  STAGE 1 — Freeze all extractors, train transformer + classifier
# ══════════════════════════════════════════════════════════════════════════════
model.freeze_backbone()

s1_opt = build_stage_optimizer(model, CONFIG["lr_stage1"])
s1_sch = build_scheduler(s1_opt, CONFIG["stage1_epochs"])
s1_es  = EarlyStopping(CONFIG["early_stop_patience"], "Stage1")

trainable_s1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Stage 1 — Trainable params: {trainable_s1:,}  (transformer + classifier only)")
print("=" * 65)

for epoch in range(1, CONFIG["stage1_epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, s1_opt, criterion, 1, epoch, CONFIG["stage1_epochs"])
    val_loss, y_true, y_prob = validate(model, val_loader, criterion)
    m  = _metrics(y_true, y_prob)
    lr = s1_opt.param_groups[0]["lr"]

    _print_epoch(1, epoch, CONFIG["stage1_epochs"], train_loss, val_loss, m, lr)
    history.append({"stage": 1, "epoch": epoch, "train_loss": train_loss,
                     "val_loss": val_loss, "lr": lr, **m})
    s1_sch.step()

    if s1_es.step(m["f1"], model):
        print(f"  Early stopping at epoch {epoch} (no F1 improvement for {CONFIG['early_stop_patience']} epochs).")
        break

s1_es.restore(model)
save_model_checkpoint(model, s1_opt, s1_sch, epoch, m, MODELS_DIR/"last_multimodal_model.pth", stage=1)
print(f"\nStage 1 complete — Best Val F1: {s1_es.best_val:.4f}")


Stage 1 — Trainable params: 560,641  (transformer + classifier only)


Stage 1  Epoch 1/10  [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

  [Val]:   0%|          | 0/1 [00:00<?, ?it/s]

  Stage 1  Ep  1/10  TrainLoss=0.5674  ValLoss=0.5189  Acc=0.714  P=0.700  R=0.875  F1=0.778  AUC=0.792  LR=1.0e-04


Stage 1  Epoch 2/10  [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

...

  [Val]:   0%|          | 0/1 [00:00<?, ?it/s]

  Stage 1  Ep  8/10  TrainLoss=0.3702  ValLoss=0.5361  Acc=0.714  P=0.833  R=0.625  F1=0.714  AUC=0.813  LR=2.1e-05


Stage 1  Epoch 9/10  [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

  [Val]:   0%|          | 0/1 [00:00<?, ?it/s]

  Stage 1  Ep  9/10  TrainLoss=0.4293  ValLoss=0.5661  Acc=0.714  P=0.833  R=0.625  F1=0.714  AUC=0.771  LR=9.5e-06
  Early stopping at epoch 9 (no F1 improvement for 5 epochs).
  ✓ Saved: last_multimodal_model.pth  (22,744.6 KB)

Stage 1 complete — Best Val F1: 0.8750


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 9.9  STAGE 2 — Unfreeze last 2 EfficientNet blocks + clinical + geo
# ══════════════════════════════════════════════════════════════════════════════
model.unfreeze_backbone(mode="partial")

s2_opt = build_stage_optimizer(model, CONFIG["lr_stage2"])
s2_sch = build_scheduler(s2_opt, CONFIG["stage2_epochs"])
s2_es  = EarlyStopping(CONFIG["early_stop_patience"], "Stage2")

trainable_s2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Stage 2 — Trainable params: {trainable_s2:,}  (last 2 EfficientNet blocks + clinical + geo + transformer)")
print("=" * 65)

for epoch in range(1, CONFIG["stage2_epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, s2_opt, criterion, 2, epoch, CONFIG["stage2_epochs"])
    val_loss, y_true, y_prob = validate(model, val_loader, criterion)
    m  = _metrics(y_true, y_prob)
    lr = s2_opt.param_groups[0]["lr"]

    _print_epoch(2, epoch, CONFIG["stage2_epochs"], train_loss, val_loss, m, lr)
    history.append({"stage": 2, "epoch": epoch, "train_loss": train_loss,
                     "val_loss": val_loss, "lr": lr, **m})
    s2_sch.step()

    if s2_es.step(m["f1"], model):
        print(f"  Early stopping at epoch {epoch}.")
        break

s2_es.restore(model)
save_model_checkpoint(model, s2_opt, s2_sch, epoch, m, MODELS_DIR/"last_multimodal_model.pth", stage=2)
print(f"\nStage 2 complete — Best Val F1: {s2_es.best_val:.4f}")


Stage 2 — Trainable params: 1,734,641  (last 2 EfficientNet blocks + clinical + geo + transformer)


Stage 2  Epoch 1/10  [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

  [Val]:   0%|          | 0/1 [00:00<?, ?it/s]

  Stage 2  Ep  1/10  TrainLoss=0.4760  ValLoss=0.5490  Acc=0.786  P=0.857  R=0.750  F1=0.800  AUC=0.813  LR=5.0e-05


Stage 2  Epoch 2/10  [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

...

  [Val]:   0%|          | 0/1 [00:00<?, ?it/s]

  Stage 2  Ep  5/10  TrainLoss=0.3684  ValLoss=0.5921  Acc=0.714  P=0.700  R=0.875  F1=0.778  AUC=0.750  LR=3.3e-05


Stage 2  Epoch 6/10  [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

  [Val]:   0%|          | 0/1 [00:00<?, ?it/s]

  Stage 2  Ep  6/10  TrainLoss=0.3695  ValLoss=0.5712  Acc=0.714  P=0.700  R=0.875  F1=0.778  AUC=0.792  LR=2.5e-05
  Early stopping at epoch 6.
  ✓ Saved: last_multimodal_model.pth  (31,938.1 KB)

Stage 2 complete — Best Val F1: 0.8000


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 9.10  STAGE 3 — End-to-end fine-tuning with gradient clipping
# ══════════════════════════════════════════════════════════════════════════════
model.unfreeze_backbone(mode="full")

s3_opt = build_stage_optimizer(model, CONFIG["lr_stage3"])
s3_sch = build_scheduler(s3_opt, CONFIG["stage3_epochs"])
s3_es  = EarlyStopping(CONFIG["early_stop_patience"], "Stage3")

trainable_s3 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Stage 3 — Trainable params: {trainable_s3:,}  (everything; grad clip={CONFIG['grad_clip']})")
print("=" * 65)

best_s3_f1 = -1.0
best_ckpt_path = MODELS_DIR / "best_multimodal_model.pth"

for epoch in range(1, CONFIG["stage3_epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, s3_opt, criterion, 3, epoch, CONFIG["stage3_epochs"])
    val_loss, y_true, y_prob = validate(model, val_loader, criterion)
    m  = _metrics(y_true, y_prob)
    lr = s3_opt.param_groups[0]["lr"]

    _print_epoch(3, epoch, CONFIG["stage3_epochs"], train_loss, val_loss, m, lr)
    history.append({"stage": 3, "epoch": epoch, "train_loss": train_loss,
                     "val_loss": val_loss, "lr": lr, **m})
    s3_sch.step()

    # Save best model whenever F1 improves
    if m["f1"] > best_s3_f1:
        best_s3_f1 = m["f1"]
        save_model_checkpoint(model, s3_opt, s3_sch, epoch, m, best_ckpt_path, stage=3)
        print(f"    ✓ New best model saved  (F1={best_s3_f1:.4f})")

    if s3_es.step(m["f1"], model):
        print(f"  Early stopping at epoch {epoch}.")
        break

s3_es.restore(model)
save_model_checkpoint(model, s3_opt, s3_sch, epoch, m, MODELS_DIR/"last_multimodal_model.pth", stage=3)
print(f"\nStage 3 complete — Best Val F1: {s3_es.best_val:.4f}")


Stage 3 — Trainable params: 4,612,797  (everything; grad clip=1.0)


Stage 3  Epoch 1/10  [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

  [Val]:   0%|          | 0/1 [00:00<?, ?it/s]

  Stage 3  Ep  1/10  TrainLoss=0.3599  ValLoss=0.5322  Acc=0.857  P=1.000  R=0.750  F1=0.857  AUC=0.854  LR=1.0e-05
  ✓ Saved: best_multimodal_model.pth  (54,585.5 KB)
    ✓ New best model saved  (F1=0.8571)


Stage 3  Epoch 2/10  [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

...

  [Val]:   0%|          | 0/1 [00:00<?, ?it/s]

  Stage 3  Ep  5/10  TrainLoss=0.4447  ValLoss=0.6085  Acc=0.500  P=0.571  R=0.500  F1=0.533  AUC=0.646  LR=6.5e-06


Stage 3  Epoch 6/10  [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

  [Val]:   0%|          | 0/1 [00:00<?, ?it/s]

  Stage 3  Ep  6/10  TrainLoss=0.3455  ValLoss=0.6085  Acc=0.571  P=0.667  R=0.500  F1=0.571  AUC=0.667  LR=5.0e-06
  Early stopping at epoch 6.
  ✓ Saved: last_multimodal_model.pth  (54,585.5 KB)

Stage 3 complete — Best Val F1: 0.8571


---
## Section 10 — Evaluation

Full evaluation on the held-out test set using the best checkpoint from Stage 3.

In [ ]:
ckpt = torch.load(
    best_ckpt_path,
    map_location="cpu",
    weights_only=False
)

print(type(ckpt))
print(ckpt.keys())

<class 'dict'>
dict_keys(['state_dict', 'metadata', 'optimizer_state', 'scheduler_state', 'epoch', 'metrics'])


In [ ]:
# ── 10.1  Load best model, run test loop ─────────────────────────────────────
print("Loading best_multimodal_model.pth for test evaluation...")
best_ckpt = torch.load(
    best_ckpt_path,
    map_location=DEVICE,
    weights_only=False
)
model.load_state_dict(best_ckpt["state_dict"])

test_loss, y_test_true, y_test_prob = test(model, test_loader, criterion)
y_test_pred = (y_test_prob >= 0.5).astype(int)

if len(y_test_true) == 0:
    raise RuntimeError("Test set is empty — y_true has length 0. Check DataLoader and dataset split.")
if len(y_test_prob) == 0:
    raise RuntimeError("No predictions generated — y_prob has length 0. Check model forward pass.")

print(f"\nTest samples: {len(y_test_true)}  |  Positive: {y_test_true.sum():.0f}  |  Test loss: {test_loss:.4f}")


Loading best_multimodal_model.pth for test evaluation...


  [Test]:   0%|          | 0/1 [00:00<?, ?it/s]


Test samples: 15  |  Positive: 9  |  Test loss: 0.3941


In [ ]:
# ── 10.2  Full metric suite ───────────────────────────────────────────────────
_test_m  = _metrics(y_test_true, y_test_prob)
_mcc     = matthews_corrcoef(y_test_true, y_test_pred)
tn, fp, fn, tp = confusion_matrix(y_test_true, y_test_pred, labels=[0,1]).ravel()
_spec    = tn/(tn+fp) if (tn+fp)>0 else 0.0

print("=" * 55)
print("  TEST SET — EVALUATION RESULTS")
print("=" * 55)
print(f"  Accuracy       : {_test_m['accuracy']:.4f}")
print(f"  Precision      : {_test_m['precision']:.4f}")
print(f"  Recall         : {_test_m['recall']:.4f}")
print(f"  Specificity    : {_spec:.4f}")
print(f"  F1 Score       : {_test_m['f1']:.4f}")
print(f"  ROC-AUC        : {_test_m['roc_auc']:.4f}")
print(f"  MCC            : {_mcc:.4f}")
print("=" * 55)


  TEST SET — EVALUATION RESULTS
  Accuracy       : 0.7333
  Precision      : 1.0000
  Recall         : 0.5556
  Specificity    : 1.0000
  F1 Score       : 0.7143
  ROC-AUC        : 0.9444
  MCC            : 0.5774


In [ ]:
import numpy as np

print("Unique y_test_true :", np.unique(y_test_true))
print("Unique y_test_pred :", np.unique(y_test_pred))

print("Number of samples :", len(y_test_true))

print("True class counts")
print(np.unique(y_test_true, return_counts=True))

print("Predicted class counts")
print(np.unique(y_test_pred, return_counts=True))

Unique y_test_true : [0. 1.]
Unique y_test_pred : [0 1]
Number of samples : 15
True class counts
(array([0., 1.], dtype=float32), array([6, 9]))
Predicted class counts
(array([0, 1]), array([10,  5]))


In [ ]:
# ── 10.3  Classification report ───────────────────────────────────────────────
from sklearn.metrics import classification_report

report = classification_report(
    y_test_true,
    y_test_pred,
    labels=[0,1],
    target_names=["Non-Anemic","Anemic"],
    zero_division=0
)

print(report)

report_df = pd.DataFrame(
    classification_report(
        y_test_true,
        y_test_pred,
        labels=[0,1],
        target_names=["Non-Anemic","Anemic"],
        output_dict=True,
        zero_division=0
    )
).transpose()

report_df.to_csv(
    OUTPUTS_DIR / "classification_report.csv"
)

print("Saved.")

              precision    recall  f1-score   support

  Non-Anemic       0.60      1.00      0.75         6
      Anemic       1.00      0.56      0.71         9

    accuracy                           0.73        15
   macro avg       0.80      0.78      0.73        15
weighted avg       0.84      0.73      0.73        15

Saved.


---
## Section 11 — Visualisations

All plots saved to `outputs/`.

In [ ]:
hist_df = pd.DataFrame(history)

def _save(fig, name):
    p = OUTPUTS_DIR / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved -> {p}")


In [ ]:
# ── 11.1  Training Loss ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colors = {1: "#2874A6", 2: "#1D7A5F", 3: "#922B21"}
for stg, grp in hist_df.groupby("stage"):
    x = grp["epoch"] + (stg - 1) * 100     # keep epochs on a single axis
    ax.plot(range(len(grp)), grp["train_loss"], color=colors[stg], linewidth=2,
            label=f"Stage {stg} Train Loss")
    ax.plot(range(len(grp)), grp["val_loss"],   color=colors[stg], linewidth=2,
            linestyle="--", alpha=0.7, label=f"Stage {stg} Val Loss")

# Stage boundary lines
_cumulative = 0
for stg, grp in hist_df.groupby("stage"):
    if stg < 3:
        _cumulative += len(grp)
        ax.axvline(_cumulative - 0.5, color="gray", linestyle=":", alpha=0.5)
    stg; grp

ax.set_xlabel("Epoch within Stage (Stage boundaries shown with dotted lines)")
ax.set_ylabel("BCEWithLogitsLoss")
ax.set_title("Training & Validation Loss — All Stages", fontweight="bold")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
_save(fig, "training_loss.png")


  Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\training_loss.png


In [ ]:
# ── 11.2  Accuracy & F1 curves ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for stg, grp in hist_df.groupby("stage"):
    xs = range(len(grp))
    axes[0].plot(xs, grp["accuracy"], color=colors[stg], linewidth=2, label=f"Stage {stg}")
    axes[1].plot(xs, grp["f1"],       color=colors[stg], linewidth=2, label=f"Stage {stg}")

for ax, title, ylabel in zip(axes,
                               ["Validation Accuracy", "Validation F1"],
                               ["Accuracy", "F1 Score"]):
    ax.set_title(title, fontweight="bold"); ax.set_ylabel(ylabel)
    ax.set_xlabel("Epoch"); ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
_save(fig, "accuracy_f1_curves.png")


  Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\accuracy_f1_curves.png


In [ ]:
# ── 11.3  Learning rate schedule ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
for stg, grp in hist_df.groupby("stage"):
    ax.plot(range(len(grp)), grp["lr"], color=colors[stg], linewidth=2, label=f"Stage {stg}")
ax.set_yscale("log"); ax.set_title("CosineAnnealingLR Schedule", fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Learning Rate"); ax.legend(); ax.grid(alpha=0.3)
_save(fig, "lr_schedule.png")


  Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\lr_schedule.png


In [ ]:
# ── 11.4  Confusion Matrix ─────────────────────────────────────────────────────
cm = confusion_matrix(y_test_true, y_test_pred, labels=[0,1])
fig, ax = plt.subplots(figsize=(6, 5.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0,1]); ax.set_xticklabels(["Non-Anemic","Anemic"])
ax.set_yticks([0,1]); ax.set_yticklabels(["Non-Anemic","Anemic"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — AnemiaFusionNet", fontweight="bold")
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center", fontsize=16,
                fontweight="bold", color="white" if cm[i,j]>thresh else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
_save(fig, "confusion_matrix.png")


  Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\confusion_matrix.png


In [ ]:
# ── 11.5  ROC Curve ───────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test_true, y_test_prob)
fig, ax = plt.subplots(figsize=(7, 6.5))
ax.plot(fpr, tpr, color="#922B21", linewidth=2.5,
        label=f"AnemiaFusionNet  AUC = {_test_m['roc_auc']:.3f}")
ax.plot([0,1],[0,1], color="#888", linestyle="--", linewidth=1.2, label="Random")
ax.fill_between(fpr, tpr, alpha=0.08, color="#922B21")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — AnemiaFusionNet", fontweight="bold")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)
plt.tight_layout()
_save(fig, "roc_curve.png")


  Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\roc_curve.png


In [ ]:
# ── 11.6  Precision-Recall Curve ──────────────────────────────────────────────
prec_vals, rec_vals, _ = precision_recall_curve(y_test_true, y_test_prob)
baseline = y_test_true.sum() / len(y_test_true)
fig, ax = plt.subplots(figsize=(7, 6.5))
ax.plot(rec_vals, prec_vals, color="#1D7A5F", linewidth=2.5,
        label=f"AnemiaFusionNet  F1={_test_m['f1']:.3f}")
ax.axhline(baseline, color="#888", linestyle="--", linewidth=1.2,
           label=f"Baseline (prevalence={baseline:.3f})")
ax.fill_between(rec_vals, prec_vals, alpha=0.08, color="#1D7A5F")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve — AnemiaFusionNet", fontweight="bold")
ax.legend(loc="lower left"); ax.grid(alpha=0.3)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)
plt.tight_layout()
_save(fig, "pr_curve.png")


  Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\pr_curve.png


---
## Section 12 — Final Verification

Programmatic sanity checks on weights, features, predictions, and probabilities.

In [ ]:
print("=" * 60)
print("  SECTION 12 — FINAL VERIFICATION")
print("=" * 60)

errors = []

# ── 12.1  No NaN weights ──────────────────────────────────────────────────────
for component, name in [
    (model.image_ext,    "ImageFeatureExtractor"),
    (model.clinical_ext, "ClinicalFeatureExtractor"),
    (model.geo_ext,      "GeoRiskFeatureExtractor"),
    (model.transformer,  "MultiModalTransformer"),
]:
    bad = [(n, p) for n, p in component.named_parameters() if torch.isnan(p).any()]
    if bad:
        errors.append(f"NaN weights in {name}: {[n for n,_ in bad]}")
    else:
        print(f"  ✓ No NaN weights   — {name}")

# ── 12.2  No NaN features / predictions / probabilities on test set ───────────
model.eval()
_sample_batch = next(iter(test_loader))
_imgs, _clin, _geo, _lbl = [x.to(DEVICE) for x in _sample_batch]

with torch.no_grad():
    _img_f  = model.image_ext(_imgs)
    _clin_f = model.clinical_ext(_clin)
    _geo_f  = model.geo_ext(_geo)
    _logits = model.transformer(_img_f, _clin_f, _geo_f)
    _probs  = torch.sigmoid(_logits)

for name, tensor in [
    ("image features",    _img_f),
    ("clinical features", _clin_f),
    ("geo features",      _geo_f),
    ("logits",            _logits),
    ("probabilities",     _probs),
]:
    if torch.isnan(tensor).any():
        errors.append(f"NaN detected in {name}.")
    else:
        print(f"  ✓ No NaN           — {name}")

# ── 12.3  Non-empty predictions ───────────────────────────────────────────────
if len(y_test_true) == 0:
    errors.append("y_true is empty (length 0). Check test DataLoader.")
else:
    print(f"  ✓ len(y_true) = {len(y_test_true)}")

if len(y_test_prob) == 0:
    errors.append("y_prob is empty (length 0). Check model forward pass.")
else:
    print(f"  ✓ len(y_prob) = {len(y_test_prob)}")

# ── 12.4  Probability range ───────────────────────────────────────────────────
if not (0.0 <= y_test_prob.min() and y_test_prob.max() <= 1.0):
    errors.append(f"Probabilities out of [0,1]: min={y_test_prob.min():.4f}, max={y_test_prob.max():.4f}")
else:
    print(f"  ✓ Probability range — [{y_test_prob.min():.3f}, {y_test_prob.max():.3f}]")

# ── 12.5  Output files ────────────────────────────────────────────────────────
expected_outputs = [
    "training_loss.png", "accuracy_f1_curves.png", "lr_schedule.png",
    "confusion_matrix.png", "roc_curve.png", "pr_curve.png",
    "classification_report.csv",
]
for fname in expected_outputs:
    p = OUTPUTS_DIR / fname
    if p.exists():
        print(f"  ✓ Output saved     — {fname}")
    else:
        errors.append(f"Expected output not found: {fname}")

# ── 12.6  Checkpoint files ────────────────────────────────────────────────────
for ckpt in ["best_multimodal_model.pth", "last_multimodal_model.pth"]:
    p = MODELS_DIR / ckpt
    if p.exists():
        print(f"  ✓ Checkpoint saved — {ckpt}  ({p.stat().st_size/1024:,.1f} KB)")
    else:
        errors.append(f"Checkpoint not found: {ckpt}")

# ── 12.7  Summary ─────────────────────────────────────────────────────────────
print()
if errors:
    print("VERIFICATION FAILED — the following issues were detected:")
    for e in errors:
        print(f"  ✗ {e}")
    raise RuntimeError("\n".join(errors))
else:
    print("All verification checks passed ✓")


  SECTION 12 — FINAL VERIFICATION
  ✓ No NaN weights   — ImageFeatureExtractor
  ✓ No NaN weights   — ClinicalFeatureExtractor
  ✓ No NaN weights   — GeoRiskFeatureExtractor
  ✓ No NaN weights   — MultiModalTransformer
  ✓ No NaN           — image features
  ✓ No NaN           — clinical features
  ✓ No NaN           — geo features
  ✓ No NaN           — logits
  ✓ No NaN           — probabilities
  ✓ len(y_true) = 15
  ✓ len(y_prob) = 15
  ✓ Probability range — [0.056, 0.666]
  ✓ Output saved     — training_loss.png
  ✓ Output saved     — accuracy_f1_curves.png
  ✓ Output saved     — lr_schedule.png
  ✓ Output saved     — confusion_matrix.png
  ✓ Output saved     — roc_curve.png
  ✓ Output saved     — pr_curve.png
  ✓ Output saved     — classification_report.csv
  ✓ Checkpoint saved — best_multimodal_model.pth  (54,585.5 KB)
  ✓ Checkpoint saved — last_multimodal_model.pth  (54,585.5 KB)

All verification checks passed ✓


In [ ]:
# ── 12.8  Final summary ───────────────────────────────────────────────────────
divider = "=" * 55
print(divider)
print("       PHASE 5 — TRAINING STRATEGY COMPLETE")
print(divider)
print()
print("  Stages completed:")
print(f"  Stage 1 — Best Val F1 : {s1_es.best_val:.4f}")
print(f"  Stage 2 — Best Val F1 : {s2_es.best_val:.4f}")
print(f"  Stage 3 — Best Val F1 : {s3_es.best_val:.4f}")
print()
print("  Test Set Metrics:")
print(f"    Accuracy   : {_test_m['accuracy']:.4f}")
print(f"    Precision  : {_test_m['precision']:.4f}")
print(f"    Recall     : {_test_m['recall']:.4f}")
print(f"    Specificity: {_spec:.4f}")
print(f"    F1         : {_test_m['f1']:.4f}")
print(f"    ROC-AUC    : {_test_m['roc_auc']:.4f}")
print(f"    MCC        : {_mcc:.4f}")
print()
print("  Saved checkpoints:")
for _f in sorted(MODELS_DIR.glob("*.pth")):
    print(f"    {_f.name:<42} ({_f.stat().st_size/1024:,.1f} KB)")
print()
print("  Output visualisations:")
for _f in sorted(OUTPUTS_DIR.glob("*")):
    if _f.is_file():
        print(f"    {_f.name}")
print()
print(divider)
print("  Next: Phase 6 — Evaluation & Deployment")
print(divider)


       PHASE 5 — TRAINING STRATEGY COMPLETE

  Stages completed:
  Stage 1 — Best Val F1 : 0.8750
  Stage 2 — Best Val F1 : 0.8000
  Stage 3 — Best Val F1 : 0.8571

  Test Set Metrics:
    Accuracy   : 0.7333
    Precision  : 1.0000
    Recall     : 0.5556
    Specificity: 1.0000
    F1         : 0.7143
    ROC-AUC    : 0.9444
    MCC        : 0.5774

  Saved checkpoints:
    best_multimodal_model.pth                  (54,585.5 KB)
    clinical_feature_extractor.pth             (169.2 KB)
    geo_feature_extractor.pth                  (11.2 KB)
    image_feature_extractor.pth                (15,946.4 KB)
    last_multimodal_model.pth                  (54,585.5 KB)
    multimodal_transformer.pth                 (2,203.9 KB)

  Output visualisations:
    accuracy_f1_curves.png
    classification_report.csv
    confusion_matrix.png
    lr_schedule.png
    phase3_model_architecture.png
    phase4_transformer_architecture.png
    pr_curve.png
    preprocessing_analytics_report.png
    roc_c

In [4]:
print("Train:", len(train_loader.dataset))
print("Validation:", len(val_loader.dataset))
print("Test:", len(test_loader.dataset))

NameError: name 'train_loader' is not defined